<a href="https://colab.research.google.com/github/serahnjogu-new/AI4EAC-Liquidity-Stress-Early-Warning-Challenge/blob/main/liquidity_stress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 194.9 kB/s eta 0:00:00


In [9]:
# =============================================================================
# AI4EAC Liquidity Stress Early Warning — Refined 0.80+ Pipeline
# Key Improvements: Behavioral Deltas, Target Encoding, Ridge Stacking
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import linregress

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING (Updated with your specific paths)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("LOADING DATA FROM CUSTOM PATHS")
print("=" * 70)

try:
    # Using the exact paths from your screenshot
    train = pd.read_csv('/content/Train (3).csv')
    test  = pd.read_csv('/content/Test (3).csv')
    sub   = pd.read_csv('/content/SampleSubmission (2).csv')
    print(f"✅ Successfully loaded Train ({train.shape}) and Test ({test.shape})")
except Exception as e:
    print(f"❌ ERROR: Could not find the files. Details: {e}")
    import sys
    sys.exit()

# ─────────────────────────────────────────────────────────────────────────────
# 2. CONFIG & METRICS
# ─────────────────────────────────────────────────────────────────────────────
SEED     = 42
N_FOLDS  = 5
SUBS_DIR = Path("submissions")
SUBS_DIR.mkdir(exist_ok=True)

TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"
W_LOGLOSS, W_AUC = 0.6, 0.4

np.random.seed(SEED)

def composite_score(y_true, y_prob):
    y_prob = np.clip(y_prob, 1e-15, 1 - 1e-15)
    ll  = log_loss(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    return -W_LOGLOSS * ll + W_AUC * auc

# ─────────────────────────────────────────────────────────────────────────────
# 3. FIXED CV RUNNER (Prevents SyntaxErrors)
# ─────────────────────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def run_cv(model_fn, name, X, y, X_test):
    print(f"\n  [{name}]  {N_FOLDS}-fold CV...")
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[va_idx],  y[va_idx]

        oof_p, test_p = model_fn(X_tr, y_tr, X_va, y_va, X_test)

        oof_preds[va_idx]  = oof_p
        test_preds        += test_p / N_FOLDS

        print(f"    fold {fold}: composite={composite_score(y_va, oof_p):+.5f}")

    return oof_preds, test_preds, composite_score(y, oof_preds)
# 0.80+ FEATURE ENGINEERING (Added to your existing pipeline)
# ─────────────────────────────────────────────────────────────────────────────
def add_behavioral_boosters(df):
    """Calculates sudden drops in balance or spikes in spending"""
    df = df.copy()

    # 1. Monthly Balance Velocity (M1 vs M2)
    if 'avg_daily_balance_M1' in df.columns and 'avg_daily_balance_M2' in df.columns:
        df['bal_velocity_recent'] = df['avg_daily_balance_M1'] / (df['avg_daily_balance_M2'] + 1e-9)

    # 2. Total Outflow Spike
    # Summing all major outflow types for M1 vs M2
    out_types = ["cashout", "p2p_sent", "paybill"]
    m1_cols = [f"m1_{t}_total_value" for t in out_types if f"m1_{t}_total_value" in df.columns]
    m2_cols = [f"m2_{t}_total_value" for t in out_types if f"m2_{t}_total_value" in df.columns]

    if m1_cols and m2_cols:
        df['total_outflow_spike'] = df[m1_cols].sum(axis=1) / (df[m2_cols].sum(axis=1) + 1e-9)

    return df

# Apply the new features
train = add_behavioral_boosters(train)
test  = add_behavioral_boosters(test)


# ─────────────────────────────────────────────────────────────────────────────
# CORE FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
MONTHS = list(range(1, 7))
TX_TYPES = {
    "paybill":      ["volume", "total_value"],
    "merchantpay":  ["volume", "total_value"],
    "bankin":       ["volume", "total_value"],
    "p2p_sent":     ["volume", "total_value"],
    "p2p_received": ["volume", "total_value"],
    "cashin":       ["volume", "total_value"],
    "cashout":      ["volume", "total_value"],
}

def _slope(arr_row, x):
    mask = ~np.isnan(arr_row)
    if mask.sum() < 2: return np.nan
    return linregress(x[mask], arr_row[mask]).slope

def add_behavioral_features(df):
    """Detects sudden shifts in user behavior (The 0.80+ Key)"""
    df = df.copy()

    # 1. Volatility: High fluctuation in balance often precedes stress
    bal_cols = [f"avg_daily_balance_M{m}" for m in MONTHS if f"avg_daily_balance_M{m}" in df.columns]
    if bal_cols:
        df['bal_cv'] = df[bal_cols].std(axis=1) / (df[bal_cols].mean(axis=1) + 1e-9)
        df['bal_m1_to_mean'] = df['avg_daily_balance_M1'] / (df[bal_cols].mean(axis=1) + 1e-9)

    # 2. Velocity: Sudden bursts of spending in M1 vs recent history
    for tx in ["cashout", "p2p_sent", "paybill"]:
        m1, m2, m3 = f"m1_{tx}_total_value", f"m2_{tx}_total_value", f"m3_{tx}_total_value"
        if all(c in df.columns for c in [m1, m2, m3]):
            df[f'{tx}_spike'] = df[m1] / ((df[m2] + df[m3])/2 + 1e-9)

    # 3. Activity Erosion: Count months with zero balance
    df['months_at_zero'] = (df[bal_cols] == 0).sum(axis=1)
    return df

def engineer_features(df):
    # This combines your original logic with the new behavioral features
    # (Assuming helper functions add_temporal, add_balance, add_ratio are present)
    # For brevity, I am showing the high-level call:
    df = add_behavioral_features(df)
    # ... [Your previous helper functions go here] ...
    return df

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING & STACKING
# ─────────────────────────────────────────────────────────────────────────────
# Load data, handle NAs, and Target Encode Categoricals
# Then run the CV loop...

def run_cv(model_fn, name, X, y, X_test):
    print(f"  [{name}] Training...")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds, test_preds = np.zeros(len(y)), np.zeros(len(X_test))
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[va_idx], y[va_idx]
        p_va, p_te = model_fn(X_tr, y_tr, X_va, y_va, X_test)
        oof_preds[va_idx] = p_va
        test_preds += p_te / N_FOLDS
    return oof_preds, test_preds

# Updated Model: LightGBM with Regularization
def lgbm_fn(X_tr, y_tr, X_va, y_va, X_te):
    params = {
        'learning_rate': 0.01, 'num_leaves': 63, 'max_depth': 7,
        'scale_pos_weight': (y_tr==0).sum()/(y_tr==1).sum() * 0.85,
        'lambda_l1': 0.2, 'lambda_l2': 1.5, 'objective': 'binary', 'verbose': -1
    }
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    model = lgb.train(params, dtrain, num_boost_round=5000, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
    return model.predict(X_va), model.predict(X_te)

# ─────────────────────────────────────────────────────────────────────────────
# FINAL ENSEMBLE EXECUTION
# ─────────────────────────────────────────────────────────────────────────────
# Run individual models, collect results, then:

def finalize_stacking(results, y, test_ids):
    print("\nExecuting Ridge Stacking...")
    oof_base = np.column_stack([results[n]["oof"] for n in results])
    test_base = np.column_stack([results[n]["test"] for n in results])

    meta_model = Ridge(alpha=1.0)
    meta_model.fit(oof_base, y)

    final_test = np.clip(meta_model.predict(test_base), 0.001, 0.999)

    pd.DataFrame({
        "ID": test_ids,
        "TargetLogLoss": final_test,
        "TargetRAUC": final_test
    }).to_csv(SUBS_DIR / "sub_stacked_ensemble.csv", index=False)
    print("Submission saved to sub_stacked_ensemble.csv")

LOADING DATA FROM CUSTOM PATHS
✅ Successfully loaded Train ((40000, 184)) and Test ((30000, 183))


In [10]:
# =============================================================================
# AI4EAC Liquidity Stress Early Warning — Full Competition Pipeline
# Evaluation: 60% LogLoss + 40% ROC-AUC (composite, higher = better)
# Submission format: identical probabilities in TargetLogLoss & TargetRAUC
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import linregress

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED     = 42
N_FOLDS  = 5
DATA_DIR = Path("Data")
SUBS_DIR = Path("submissions")
SUBS_DIR.mkdir(exist_ok=True)

TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"

# Metric weights (mirrors leaderboard)
W_LOGLOSS = 0.6
W_AUC     = 0.4

np.random.seed(SEED)


# ─────────────────────────────────────────────────────────────────────────────
# COMPOSITE METRIC  (higher = better)
# ─────────────────────────────────────────────────────────────────────────────
def composite_score(y_true, y_prob):
    ll  = log_loss(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    return -W_LOGLOSS * ll + W_AUC * auc


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING (Updated with your specific paths)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("LOADING DATA FROM CUSTOM PATHS")
print("=" * 70)

try:
    # Using the exact paths from your screenshot
    train = pd.read_csv('/content/Train (3).csv')
    test  = pd.read_csv('/content/Test (3).csv')
    sub   = pd.read_csv('/content/SampleSubmission (2).csv')
    print(f"✅ Successfully loaded Train ({train.shape}) and Test ({test.shape})")
except Exception as e:
    print(f"❌ ERROR: Could not find the files. Details: {e}")
    print(f"Target distribution:\n{train[TARGET].value_counts(normalize=True).round(4)}")

    import sys
    sys.exit()
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

MONTHS = list(range(1, 7))   # M1 = most recent, M6 = oldest
TX_TYPES = {
    "paybill":      ["volume", "total_value", "highest_amount", "companies"],
    "merchantpay":  ["volume", "total_value", "highest_amount", "merchants"],
    "bankin":       ["volume", "total_value", "highest_amount", "counterparties"],
    "p2p_sent":     ["volume", "total_value", "highest_amount", "counterparties"],
    "p2p_received": ["volume", "total_value", "highest_amount", "counterparties"],
    "cashin":       ["volume", "total_value", "highest_amount", "agents"],
    "cashout":      ["volume", "total_value", "highest_amount", "agents"],
}


def _slope(arr_row, x):
    """OLS slope; returns NaN when insufficient data."""
    mask = ~np.isnan(arr_row)
    if mask.sum() < 2:
        return np.nan
    return linregress(x[mask], arr_row[mask]).slope


def _col(tx, metric, m):
    return f"m{m}_{tx}_{metric}"


def add_temporal_features(df):
    df = df.copy()
    cols_set = set(df.columns)
    feats = {}

    # x-axis for slope: M6→M1 mapped to 6→1 (positive slope = growing toward M1)
    x_full = np.arange(len(MONTHS), 0, -1, dtype=float)

    for tx, metrics in TX_TYPES.items():
        for metric in metrics:
            present = [_col(tx, metric, m) for m in MONTHS if _col(tx, metric, m) in cols_set]
            if not present:
                continue

            arr = df[present].values.astype(float)
            key = f"{tx}_{metric}"

            feats[f"{key}_mean"] = np.nanmean(arr, axis=1)
            feats[f"{key}_std"]  = np.nanstd(arr, axis=1)
            feats[f"{key}_sum"]  = np.nansum(arr, axis=1)
            feats[f"{key}_max"]  = np.nanmax(arr, axis=1)

            # M1 relative to historical mean (M2–M6)
            if arr.shape[1] >= 2:
                hist = np.nanmean(arr[:, 1:], axis=1)
                feats[f"{key}_m1_vs_hist"] = arr[:, 0] / (hist + 1e-9)

            # Linear slope across the available months
            n = arr.shape[1]
            x_sub = x_full[:n]
            feats[f"{key}_slope"] = [_slope(arr[i], x_sub) for i in range(len(df))]

    return pd.concat([df, pd.DataFrame(feats, index=df.index)], axis=1)


def add_balance_features(df):
    df = df.copy()
    bal_cols = [f"avg_daily_balance_M{m}" for m in MONTHS if f"avg_daily_balance_M{m}" in df.columns]
    if not bal_cols:
        return df

    arr = df[bal_cols].values.astype(float)
    df["balance_mean"] = np.nanmean(arr, axis=1)
    df["balance_std"]  = np.nanstd(arr, axis=1)
    df["balance_min"]  = np.nanmin(arr, axis=1)
    df["balance_cv"]   = df["balance_std"] / (df["balance_mean"].abs() + 1e-9)

    b1, b6 = "avg_daily_balance_M1", "avg_daily_balance_M6"
    if b1 in df.columns and b6 in df.columns:
        df["balance_change"]     = df[b1] - df[b6]
        df["balance_pct_change"] = df["balance_change"] / (df[b6].abs() + 1e-9)

    x_sub = np.arange(len(bal_cols), 0, -1, dtype=float)
    df["balance_slope"] = [_slope(arr[i], x_sub) for i in range(len(df))]

    return df


def add_ratio_features(df):
    df = df.copy()
    cols_set = set(df.columns)

    inflow_types  = ["p2p_received", "cashin", "bankin"]
    outflow_types = ["p2p_sent", "cashout", "paybill", "merchantpay"]

    for m in MONTHS:
        def _sum_tv(types):
            vals = [df[_col(t, "total_value", m)]
                    for t in types if _col(t, "total_value", m) in cols_set]
            return sum(vals) if vals else pd.Series(0, index=df.index)

        inf  = _sum_tv(inflow_types)
        outf = _sum_tv(outflow_types)
        df[f"m{m}_net_flow"]        = inf - outf
        df[f"m{m}_liquidity_ratio"] = inf / (outf + 1e-9)

    # M1 net flow vs average of M2–M6
    net_cols = [f"m{m}_net_flow" for m in MONTHS if f"m{m}_net_flow" in df.columns]
    if len(net_cols) >= 2:
        hist_net = df[net_cols[1:]].mean(axis=1)
        df["net_flow_m1_vs_hist"] = df[net_cols[0]] - hist_net

    # Cashout acceleration: M1 vs M2
    c1 = _col("cashout", "total_value", 1)
    c2 = _col("cashout", "total_value", 2)
    if c1 in cols_set and c2 in cols_set:
        df["cashout_accel"] = (df[c1] - df[c2]) / (df[c2] + 1e-9)

    # Balance-to-cashout pressure
    b1 = "avg_daily_balance_M1"
    if b1 in cols_set and c1 in cols_set:
        df["balance_to_cashout"] = df[b1] / (df[c1] + 1e-9)

    return df


def engineer_features(df):
    t0 = time.time()
    df = add_temporal_features(df)
    df = add_balance_features(df)
    df = add_ratio_features(df)
    print(f"  Done in {time.time()-t0:.1f}s  →  shape {df.shape}")
    return df


print("Engineering train features...")
train = engineer_features(train)
print("Engineering test features...")
test  = engineer_features(test)


# ─────────────────────────────────────────────────────────────────────────────
# PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PREPROCESSING")
print("=" * 70)

CAT_COLS = ["gender", "region", "smartphone", "segment", "earning_pattern"]

for col in CAT_COLS:
    if col not in train.columns:
        continue
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    print(f"  Encoded '{col}': {len(le.classes_)} classes")

feat_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]

X      = train[feat_cols].copy()
y      = train[TARGET].values.astype(int)
X_test = test[feat_cols].reindex(columns=feat_cols, fill_value=0).copy()

# Clip infinities, impute with train median
X      = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
medians = X.median()
X      = X.fillna(medians)
X_test = X_test.fillna(medians)

print(f"\nFeature matrix : {X.shape}")
print(f"Positive rate  : {y.mean():.4f}")

pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
print(f"pos_weight     : {pos_weight:.2f}")


# ─────────────────────────────────────────────────────────────────────────────
# ADVERSARIAL VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ADVERSARIAL VALIDATION  (train=0 vs test=1)")
print("=" * 70)

adv_X  = pd.concat([X, X_test], axis=0).reset_index(drop=True)
adv_y  = np.array([0] * len(X) + [1] * len(X_test))
adv_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
adv_aucs = []

for tr_i, va_i in adv_cv.split(adv_X, adv_y):
    m = lgb.LGBMClassifier(n_estimators=100, max_depth=4, random_state=SEED,
                            n_jobs=-1, verbose=-1)
    m.fit(adv_X.iloc[tr_i], adv_y[tr_i])
    adv_aucs.append(roc_auc_score(adv_y[va_i], m.predict_proba(adv_X.iloc[va_i])[:, 1]))

adv_auc = np.mean(adv_aucs)
flag = "⚠ SIGNIFICANT SHIFT" if adv_auc > 0.70 else "OK"
print(f"  Adversarial AUC = {adv_auc:.4f}  [{flag}]")


# ─────────────────────────────────────────────────────────────────────────────
# CV RUNNER
# ─────────────────────────────────────────────────────────────────────────────
# model_fn signature: (X_tr, y_tr, X_va, y_va, X_te) -> (oof_proba, test_proba)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def run_cv(model_fn, name, X, y, X_test):
    print(f"\n  [{name}]  {N_FOLDS}-fold CV...")
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[va_idx],  y[va_idx]

        oof_p, test_p = model_fn(X_tr, y_tr, X_va, y_va, X_test)

        oof_preds[va_idx]  = oof_p
        test_preds        += test_p / N_FOLDS

        s = composite_score(y_va, oof_p)
        print(f"    fold {fold}: composite={s:+.5f}  "
              f"ll={log_loss(y_va, oof_p):.5f}  "
              f"auc={roc_auc_score(y_va, oof_p):.5f}")

    cv = composite_score(y, oof_preds)
    print(f"  [{name}] OOF → composite={cv:+.5f}  "
          f"ll={log_loss(y, oof_preds):.5f}  "
          f"auc={roc_auc_score(y, oof_preds):.5f}")
    return oof_preds, test_preds, cv


# ─────────────────────────────────────────────────────────────────────────────
# MODEL FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def lgbm_fn(X_tr, y_tr, X_va, y_va, X_te):
    params = dict(
        objective="binary",
        metric="binary_logloss",
        learning_rate=0.03,
        num_leaves=127,
        min_child_samples=30,
        feature_fraction=0.7,
        bagging_fraction=0.8,
        bagging_freq=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=pos_weight,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    booster = lgb.train(
        params, dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    return booster.predict(X_va), booster.predict(X_te)


def xgb_fn(X_tr, y_tr, X_va, y_va, X_te):
    params = dict(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.7,
        scale_pos_weight=pos_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric="logloss",
        early_stopping_rounds=100,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]


def cat_fn(X_tr, y_tr, X_va, y_va, X_te):
    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3.0,
        scale_pos_weight=pos_weight,
        eval_metric="Logloss",
        early_stopping_rounds=100,
        random_seed=SEED,
        verbose=0,
        thread_count=-1,
    )
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]


def hgb_fn(X_tr, y_tr, X_va, y_va, X_te):
    model = HistGradientBoostingClassifier(
        max_iter=600,
        learning_rate=0.05,
        max_depth=6,
        min_samples_leaf=30,
        l2_regularization=1.0,
        class_weight={0: 1.0, 1: pos_weight},
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=SEED,
        verbose=0,
    )
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]


def rf_fn(X_tr, y_tr, X_va, y_va, X_te):
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=16,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]


def lr_fn(X_tr, y_tr, X_va, y_va, X_te):
    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(X_tr)
    model  = LogisticRegression(
        C=0.1, max_iter=1000, class_weight="balanced",
        solver="lbfgs", random_state=SEED, n_jobs=-1,
    )
    model.fit(Xtr_s, y_tr)
    return (model.predict_proba(scaler.transform(X_va))[:, 1],
            model.predict_proba(scaler.transform(X_te))[:, 1])


def mlp_fn(X_tr, y_tr, X_va, y_va, X_te):
    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(X_tr)
    model  = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=SEED,
    )
    model.fit(Xtr_s, y_tr)
    return (model.predict_proba(scaler.transform(X_va))[:, 1],
            model.predict_proba(scaler.transform(X_te))[:, 1])


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN ALL MODELS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("TRAINING MODELS")
print("=" * 70)

MODELS = {
    "lgbm": lgbm_fn,
    "xgb":  xgb_fn,
    "cat":  cat_fn,
    "hgb":  hgb_fn,
    "rf":   rf_fn,
    "lr":   lr_fn,
    "mlp":  mlp_fn,
}

results = {}
for name, fn in MODELS.items():
    t0 = time.time()
    try:
        oof, test_pred, score = run_cv(fn, name, X, y, X_test)
        results[name] = {"oof": oof, "test": test_pred, "score": score}
        elapsed = time.time() - t0
        print(f"  ✓ {name} completed in {elapsed:.1f}s")
    except Exception as e:
        print(f"  ✗ {name} FAILED: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# INDIVIDUAL SUBMISSIONS
# ─────────────────────────────────────────────────────────────────────────────
def make_submission(ids, proba, path):
    pd.DataFrame({
        "ID":            ids,
        "TargetLogLoss": proba,
        "TargetRAUC":    proba,
    }).to_csv(path, index=False)
    print(f"  Saved: {path}")


test_ids = test[ID_COL].values

print("\n" + "=" * 70)
print("INDIVIDUAL SUBMISSIONS")
print("=" * 70)
for name, res in results.items():
    make_submission(test_ids, res["test"], SUBS_DIR / f"sub_{name}.csv")


# ─────────────────────────────────────────────────────────────────────────────
# LEADERBOARD SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("CV LEADERBOARD  (composite = -0.6×LogLoss + 0.4×AUC, higher = better)")
print("=" * 70)

ranked = sorted(results.items(), key=lambda x: x[1]["score"], reverse=True)

print(f"  {'Rank':<5} {'Model':<8} {'Composite':>12} {'LogLoss':>10} {'AUC':>10}")
print("  " + "-" * 47)
for rank, (name, res) in enumerate(ranked, 1):
    ll  = log_loss(y, res["oof"])
    auc = roc_auc_score(y, res["oof"])
    print(f"  #{rank:<4} {name:<8} {res['score']:>+12.5f} {ll:>10.5f} {auc:>10.5f}")


# ─────────────────────────────────────────────────────────────────────────────
# ENSEMBLE — TOP 3
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ENSEMBLE  (top-3 by CV composite)")
print("=" * 70)

top3 = ranked[:3]
top3_names = [n for n, _ in top3]
print(f"  Members: {top3_names}")

# Strategy A: score-proportional weights (all scores shifted to be positive)
scores_arr    = np.array([r["score"] for _, r in top3])
scores_pos    = scores_arr - scores_arr.min() + 1e-6
weights_prop  = scores_pos / scores_pos.sum()

ens_oof_w  = sum(w * results[n]["oof"]  for w, (n, _) in zip(weights_prop, top3))
ens_test_w = sum(w * results[n]["test"] for w, (n, _) in zip(weights_prop, top3))
score_w    = composite_score(y, ens_oof_w)

# Strategy B: simple mean
ens_oof_m  = np.mean([results[n]["oof"]  for n, _ in top3], axis=0)
ens_test_m = np.mean([results[n]["test"] for n, _ in top3], axis=0)
score_m    = composite_score(y, ens_oof_m)

print(f"  Weighted ensemble  OOF composite = {score_w:+.5f}")
print(f"  Mean ensemble      OOF composite = {score_m:+.5f}")

if score_w >= score_m:
    best_ens_test, best_ens_label = ens_test_w, "ensemble_weighted_top3"
    print(f"  → Using: weighted ensemble")
else:
    best_ens_test, best_ens_label = ens_test_m, "ensemble_mean_top3"
    print(f"  → Using: mean ensemble")

make_submission(test_ids, best_ens_test, SUBS_DIR / f"sub_{best_ens_label}.csv")

# Also save both for safety
make_submission(test_ids, ens_test_w, SUBS_DIR / "sub_ensemble_weighted_top3.csv")
make_submission(test_ids, ens_test_m, SUBS_DIR / "sub_ensemble_mean_top3.csv")


# ─────────────────────────────────────────────────────────────────────────────
# SAVE OOF PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
oof_df = pd.DataFrame({"ID": train[ID_COL], TARGET: y})
for name, res in results.items():
    oof_df[f"oof_{name}"] = res["oof"]
oof_df.to_csv(SUBS_DIR / "oof_predictions.csv", index=False)
print(f"\nOOF predictions → {SUBS_DIR}/oof_predictions.csv")


# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print(f"\nAll submissions saved to '{SUBS_DIR}/':")
for f in sorted(SUBS_DIR.glob("sub_*.csv")):
    print(f"  {f.name}")

print("\nRecommended submission order (by CV score):")
for rank, (name, res) in enumerate(ranked, 1):
    fname = f"sub_{name}.csv"
    print(f"  #{rank}  {fname:<35}  composite={res['score']:+.5f}")
print(f"  #E  sub_{best_ens_label}.csv")

LOADING DATA FROM CUSTOM PATHS
✅ Successfully loaded Train ((40000, 184)) and Test ((30000, 183))

FEATURE ENGINEERING
Engineering train features...
  Done in 177.6s  →  shape (40000, 245)
Engineering test features...
  Done in 131.8s  →  shape (30000, 244)

PREPROCESSING
  Encoded 'gender': 2 classes
  Encoded 'region': 7 classes
  Encoded 'smartphone': 2 classes
  Encoded 'segment': 3 classes
  Encoded 'earning_pattern': 3 classes

Feature matrix : (40000, 243)
Positive rate  : 0.1500
pos_weight     : 5.67

ADVERSARIAL VALIDATION  (train=0 vs test=1)
  Adversarial AUC = 0.5016  [OK]

TRAINING MODELS

  [lgbm]  5-fold CV...
    fold 1: composite=+0.19469  ll=0.26844  auc=0.88937
    fold 2: composite=+0.18922  ll=0.27404  auc=0.88412
    fold 3: composite=+0.18350  ll=0.28000  auc=0.87875
    fold 4: composite=+0.18444  ll=0.27950  auc=0.88035
    fold 5: composite=+0.17780  ll=0.28540  auc=0.87259
  [lgbm] OOF → composite=+0.18594  ll=0.27748  auc=0.88106
  ✓ lgbm completed in 488.3s

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUBMISSION EXPORT & DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import files

print("\n" + "=" * 70)
print("GENERATING & DOWNLOADING FINAL SUBMISSION")
print("=" * 70)

# Create the final DataFrame
final_sub = pd.DataFrame({
    "ID":            test[ID_COL],
    "TargetLogLoss": best_ens_test,
    "TargetRAUC":    best_ens_test
})

# Save to the local content folder for easy downloading
file_name = "final_competition_submission.csv"
final_sub.to_csv(file_name, index=False)

print(f"✅ Success! File '{file_name}' created locally.")

# Trigger the Colab download prompt
files.download(file_name)


GENERATING & DOWNLOADING FINAL SUBMISSION
✅ Success! File 'final_competition_submission.csv' created locally.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>